# Simulation core

This notebook defines the coarse-grained total-transposable-element-copy-number (TCN) model used for Figure 2, Figure 5a, and Supplementary Figures 2–4 and 11–14. Figure-specific notebooks load it with `%run ./simulation_core.ipynb` and only specify the parameters varied for that figure.

For the primary model, a maximum plasmid-associated copy-number capacity $n$ gives states $T_i$, $i=1,\ldots,n+1$. $T_i$ is the scaled cell density with total TCN $i$; one copy is chromosomal and $i-1$ copies are plasmid associated. The chromosome-free sensitivity uses the same state-vector length but assigns total TCN $i=0,\ldots,n$.

Within a 24-h passage,

$$
\frac{d\mathbf T}{dt}=\left(\boldsymbol\mu\odot\mathbf T+Q\mathbf T\right)\left(1-\frac{\sum_i T_i}{K}\right),
$$

where $Q$ is the adjacent-state transition generator. Interior forward and backward coefficients are $\kappa_f$ and $\kappa_b$. In the primary model, $T_1\to T_2$ uses a shared basal forward coefficient in all configurations; at the upper boundary there is no forward transition beyond the maximum state.

Relaxed growth and selected growth are

$$
\mu_i^{(-A)}=\mu_\infty+\frac{\mu_{\max}-\mu_\infty}{1+p_i/n_c},\qquad
\mu_i^{(+A)}=\mu_i^{(-A)}-\delta_A\frac{A^h}{A^h+[\gamma(1+\alpha i)]^h},
$$

where $p_i$ is plasmid-associated TCN. In the primary model, $p_i=i-1$; in the chromosome-free sensitivity, $p_i=i$ and the total TCN in the protection term is also reduced by one.

Each modeled day is followed by a 30,000-fold dilution. A fluctuating cycle has five relaxed passages and two selected passages. Periodic analyses use the convergence, survival, and numerical criteria specified below.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import os
import platform

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.stats import norm

CORE_VERSION = "2026-09-07-simulation-core-v1"
PARAMETER_MANIFEST_VERSION = "2026-09-07-final-ms-si"


def resolve_run_mode(env_var: str = "SIM_RUN_MODE") -> str:
    mode = os.environ.get(env_var, "production").strip().lower()
    if mode not in {"quick", "production"}:
        raise ValueError(f"{env_var} must be 'quick' or 'production'.")
    return mode


RUN_MODE = resolve_run_mode()
MAX_STEP_HOURS = 1.0 if RUN_MODE == "quick" else 0.5
OUTPUT_STEP_HOURS = 4.0 if RUN_MODE == "quick" else 1.0
ATOL = 1e-11
RTOL = 1e-6
NEGATIVE_STATE_TOLERANCE = -1e-8

TOTAL_DENSITY_SURVIVAL_THRESHOLD = 0.1
SUBPOPULATION_CUTOFF = 0.0

COMPOSITION_CONVERGENCE_TOLERANCE = 0.001
BOUNDARY_DENSITY_LOG_CONVERGENCE_TOLERANCE = 0.01
ENDPOINT_TCN_LOG_CONVERGENCE_TOLERANCE = 0.001
CONSECUTIVE_CONVERGED_CYCLES = 8
MIN_CONVERGENCE_CYCLES = 60
MAX_CONVERGENCE_CYCLES = 120 if RUN_MODE == "quick" else 250
ADDITIONAL_CYCLES_AFTER_CONVERGENCE = 3

PASSAGE_INTERVAL_HOURS = 24.0
PASSAGE_DILUTION_FACTOR = 30_000.0
PASSAGE_GENERATIONS = float(np.log2(PASSAGE_DILUTION_FACTOR))
RELAXED_DAYS_PER_CYCLE = 5
SELECTED_DAYS_PER_CYCLE = 2
DAYS_PER_CYCLE = RELAXED_DAYS_PER_CYCLE + SELECTED_DAYS_PER_CYCLE
TRANSIENT_CYCLES = 5

PERIODIC_INITIAL_MEAN_FRACTION_OF_PCN = 0.50
PERIODIC_INITIAL_SD_FRACTION_OF_PCN = 0.30
FIXED_LOSS_INITIAL_MEAN_FRACTION_OF_PCN = 0.80
FIXED_GAIN_INITIAL_MEAN_FRACTION_OF_PCN = 0.10
FIXED_INITIAL_SD_FRACTION_OF_PCN = 0.15

SELECTION_SHADE = "#DDF1D8"


@dataclass(frozen=True)
class ModelParameters:
    mu_max: float = 2.0
    mu_infinity: float = 0.8
    burden_scale: float = 35.0
    delta_antibiotic: float = 2.0
    gamma: float = 0.002
    alpha: float = 40.0
    hill: float = 2.0
    antibiotic_relaxed: float = 0.0
    antibiotic_selected: float = 2.0
    carrying_capacity: float = 1.0


@dataclass(frozen=True)
class TransitionParameters:
    kappa_f_off: float = 0.0075
    kappa_b_off: float = 0.030
    kappa_f_on: float = 0.100
    kappa_b_on: float = 0.120
    kappa_f_pcn: float = 0.250
    kappa_b_pcn: float = 0.220
    lower_forward_rate: float = 0.0075


@dataclass(frozen=True)
class GrowthProfile:
    name: str
    burden_scale: float
    protection_alpha: float


MODEL = ModelParameters()
TRANSITIONS = TransitionParameters()
DEFAULT_GROWTH_PROFILE = GrowthProfile("default", MODEL.burden_scale, MODEL.alpha)

CONDITION_ORDER = (
    "transposition off",
    "transposition on",
    "transposition on + PCN dynamics",
)
DISPLAY_LABELS = {
    "transposition off": "baseline",
    "transposition on": "+ transposition",
    "transposition on + PCN dynamics": "+ transposition + PCN dynamics",
}
CONDITION_COLORS = {
    "transposition off": "#C3C5DE",
    "transposition on": "#8C82BC",
    "transposition on + PCN dynamics": "#3A2A72",
}
CONDITION_LINEWIDTHS = {
    "transposition off": 1.1,
    "transposition on": 1.4,
    "transposition on + PCN dynamics": 1.7,
}

PLOT_STYLE = {
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Nimbus Sans", "DejaVu Sans"],
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "lines.solid_capstyle": "round",
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}


## State space, growth, and transitions

`chromosomal_copy=1` is the primary model. `chromosomal_copy=0` is used only for Supplementary Figure 2. Maximum PCN capacity remains the number of plasmid-associated states above the lower boundary in either case.


In [ ]:
def total_tcn_states(max_pcn: int, *, chromosomal_copy: int = 1) -> np.ndarray:
    if int(max_pcn) != max_pcn or max_pcn < 1:
        raise ValueError("max_pcn must be a positive integer.")
    if chromosomal_copy not in {0, 1}:
        raise ValueError("chromosomal_copy must be 0 or 1.")
    return np.arange(chromosomal_copy, int(max_pcn) + chromosomal_copy + 1, dtype=float)


def growth_rates(
    max_pcn: int,
    *,
    selected: bool,
    chromosomal_copy: int = 1,
    model: ModelParameters = MODEL,
    growth_profile: GrowthProfile | None = None,
) -> np.ndarray:
    total_tcn = total_tcn_states(max_pcn, chromosomal_copy=chromosomal_copy)
    plasmid_associated_tcn = total_tcn - chromosomal_copy
    profile = growth_profile or DEFAULT_GROWTH_PROFILE
    if profile.burden_scale <= 0 or profile.protection_alpha < 0:
        raise ValueError("Growth-profile parameters must be nonnegative and burden_scale > 0.")

    relaxed = model.mu_infinity + (
        (model.mu_max - model.mu_infinity)
        / (1.0 + plasmid_associated_tcn / profile.burden_scale)
    )
    antibiotic = model.antibiotic_selected if selected else model.antibiotic_relaxed
    ec_i = model.gamma * (1.0 + profile.protection_alpha * total_tcn)
    inhibition = model.delta_antibiotic * (
        antibiotic**model.hill / (antibiotic**model.hill + ec_i**model.hill)
    )
    result = relaxed - inhibition
    if not np.isfinite(result).all():
        raise FloatingPointError("The growth function produced a non-finite value.")
    return result


@dataclass(frozen=True)
class TransitionConfiguration:
    condition: str
    max_pcn: int
    setting_label: str
    kappa_f: float
    kappa_b: float
    lower_forward_rate: float


def make_configuration(
    max_pcn: int,
    *,
    condition: str,
    kappa_f: float,
    kappa_b: float,
    setting_label: str,
    lower_forward_rate: float | None = None,
) -> TransitionConfiguration:
    if condition not in CONDITION_ORDER:
        raise ValueError(f"Unknown condition: {condition}")
    if kappa_f <= 0 or kappa_b <= 0:
        raise ValueError("kappa_f and kappa_b must be positive.")
    return TransitionConfiguration(
        condition=condition,
        max_pcn=int(max_pcn),
        setting_label=setting_label,
        kappa_f=float(kappa_f),
        kappa_b=float(kappa_b),
        lower_forward_rate=float(
            TRANSITIONS.lower_forward_rate if lower_forward_rate is None else lower_forward_rate
        ),
    )


def standard_configurations(max_pcn: int, *, setting_label: str = "Figure 2 primary"):
    spec = (
        ("transposition off", TRANSITIONS.kappa_f_off, TRANSITIONS.kappa_b_off),
        ("transposition on", TRANSITIONS.kappa_f_on, TRANSITIONS.kappa_b_on),
        ("transposition on + PCN dynamics", TRANSITIONS.kappa_f_pcn, TRANSITIONS.kappa_b_pcn),
    )
    return [
        make_configuration(
            max_pcn,
            condition=condition,
            kappa_f=kappa_f,
            kappa_b=kappa_b,
            setting_label=setting_label,
        )
        for condition, kappa_f, kappa_b in spec
    ]


def transition_matrix(max_pcn: int, config: TransitionConfiguration) -> np.ndarray:
    if int(max_pcn) != config.max_pcn:
        raise ValueError("Configuration and transition matrix must use the same maximum PCN.")
    n_states = int(max_pcn) + 1
    matrix = np.zeros((n_states, n_states), dtype=float)
    for source in range(n_states):
        forward = config.kappa_f if source < max_pcn else 0.0
        backward = config.kappa_b if source > 0 else 0.0
        if source == 0:
            forward = config.lower_forward_rate
        if forward > 0:
            matrix[source + 1, source] += forward
            matrix[source, source] -= forward
        if backward > 0:
            matrix[source - 1, source] += backward
            matrix[source, source] -= backward
    return matrix


def assert_generator_valid(matrix: np.ndarray, label: str = "") -> None:
    if not np.allclose(matrix.sum(axis=0), 0.0, atol=1e-12):
        raise RuntimeError(f"Transition generator {label} does not conserve density.")
    off = matrix.copy()
    np.fill_diagonal(off, 0.0)
    if (off < -1e-14).any():
        raise RuntimeError(f"Transition generator {label} has a negative off-diagonal entry.")


## Initialization and within-passage integration

Initial compositions are normalized to the standardized post-passage total density $1/30{,}000$. The simulation is deterministic; the post-passage subpopulation cutoff is zero.


In [ ]:
def normalize_state(state: np.ndarray, total_abundance: float | None = None) -> np.ndarray:
    if total_abundance is None:
        total_abundance = MODEL.carrying_capacity / PASSAGE_DILUTION_FACTOR
    state = np.asarray(state, dtype=float)
    if np.any(state < 0) or not np.isfinite(state).all():
        raise ValueError("State weights must be finite and nonnegative.")
    total = float(state.sum())
    if total <= 0:
        raise ValueError("State weights must have positive mass.")
    return state * (float(total_abundance) / total)


def normal_initial_state(
    max_pcn: int,
    *,
    mean_fraction_of_pcn: float,
    sd_fraction_of_pcn: float,
    chromosomal_copy: int = 1,
) -> np.ndarray:
    states = total_tcn_states(max_pcn, chromosomal_copy=chromosomal_copy)
    # The nominal mean and s.d. are defined relative to n, as in Table S1.
    weights = norm.pdf(
        states,
        loc=mean_fraction_of_pcn * max_pcn,
        scale=sd_fraction_of_pcn * max_pcn,
    )
    return normalize_state(weights)


def periodic_initial_state(max_pcn: int, *, chromosomal_copy: int = 1) -> np.ndarray:
    return normal_initial_state(
        max_pcn,
        mean_fraction_of_pcn=PERIODIC_INITIAL_MEAN_FRACTION_OF_PCN,
        sd_fraction_of_pcn=PERIODIC_INITIAL_SD_FRACTION_OF_PCN,
        chromosomal_copy=chromosomal_copy,
    )


def fixed_loss_initial_state(max_pcn: int, *, chromosomal_copy: int = 1) -> np.ndarray:
    return normal_initial_state(
        max_pcn,
        mean_fraction_of_pcn=FIXED_LOSS_INITIAL_MEAN_FRACTION_OF_PCN,
        sd_fraction_of_pcn=FIXED_INITIAL_SD_FRACTION_OF_PCN,
        chromosomal_copy=chromosomal_copy,
    )


def fixed_gain_initial_state(max_pcn: int, *, chromosomal_copy: int = 1) -> np.ndarray:
    return normal_initial_state(
        max_pcn,
        mean_fraction_of_pcn=FIXED_GAIN_INITIAL_MEAN_FRACTION_OF_PCN,
        sd_fraction_of_pcn=FIXED_INITIAL_SD_FRACTION_OF_PCN,
        chromosomal_copy=chromosomal_copy,
    )


def uniform_initial_state(max_pcn: int) -> np.ndarray:
    return normalize_state(np.ones(int(max_pcn) + 1, dtype=float))


def random_initial_state(max_pcn: int, rng: np.random.Generator) -> np.ndarray:
    return normalize_state(rng.uniform(0.0, 1.0, size=int(max_pcn) + 1))


def normalized_composition(state: np.ndarray) -> np.ndarray:
    state = np.maximum(np.asarray(state, dtype=float), 0.0)
    total = float(state.sum())
    return state / total if total > 0 else np.full_like(state, np.nan)


def state_moments(state: np.ndarray, *, chromosomal_copy: int = 1) -> dict:
    tcn = total_tcn_states(len(state) - 1, chromosomal_copy=chromosomal_copy)
    p = normalized_composition(state)
    if not np.isfinite(p).all():
        return {"realized_mean_TCN": np.nan, "realized_sd_TCN": np.nan}
    mean = float(np.dot(p, tcn))
    variance = float(np.dot(p, np.square(tcn - mean)))
    return {"realized_mean_TCN": mean, "realized_sd_TCN": float(np.sqrt(variance))}


def evaluation_times(duration_hours: float = PASSAGE_INTERVAL_HOURS) -> np.ndarray:
    times = np.arange(0.0, duration_hours + OUTPUT_STEP_HOURS, OUTPUT_STEP_HOURS)
    times = times[times <= duration_hours]
    if not np.isclose(times[-1], duration_hours):
        times = np.append(times, duration_hours)
    return times


def summarize_state(state: np.ndarray, growth: np.ndarray, *, chromosomal_copy: int = 1) -> dict:
    raw = np.asarray(state, dtype=float)
    minimum_raw = float(np.min(raw))
    if minimum_raw < NEGATIVE_STATE_TOLERANCE:
        raise FloatingPointError(f"Raw state below tolerance: {minimum_raw:.3e}")
    state = np.maximum(raw, 0.0)
    density = float(state.sum())
    tcn_states = total_tcn_states(len(state) - 1, chromosomal_copy=chromosomal_copy)
    tcn_signal = float(np.dot(state, tcn_states))
    return {
        "density": density,
        "TCN_signal": tcn_signal,
        "TCN_per_density": tcn_signal / density if density > 0 else np.nan,
        "mean_intrinsic_growth": float(np.dot(state / density, growth)) if density > 0 else np.nan,
        "minimum_raw_state": minimum_raw,
    }


def ode_rhs(_time, state, growth, transitions, carrying_capacity):
    density = float(np.sum(state))
    logistic_factor = 1.0 - density / carrying_capacity
    return (growth * state + transitions @ state) * logistic_factor


def integrate_day(state, growth, transitions, *, record_trajectory: bool):
    t_eval = evaluation_times() if record_trajectory else np.array([PASSAGE_INTERVAL_HOURS])
    solution = solve_ivp(
        ode_rhs,
        (0.0, PASSAGE_INTERVAL_HOURS),
        np.asarray(state, dtype=float),
        args=(growth, transitions, MODEL.carrying_capacity),
        method="LSODA",
        t_eval=t_eval,
        max_step=MAX_STEP_HOURS,
        atol=ATOL,
        rtol=RTOL,
    )
    if not solution.success:
        raise RuntimeError(solution.message)
    if not np.isfinite(solution.y).all():
        raise FloatingPointError("Non-finite state encountered.")
    minimum_raw = float(np.min(solution.y))
    if minimum_raw < NEGATIVE_STATE_TOLERANCE:
        raise FloatingPointError(f"Raw state below tolerance: {minimum_raw:.3e}")
    return solution.t, solution.y, np.maximum(solution.y[:, -1], 0.0)


def population_is_extinct(state: np.ndarray) -> bool:
    return float(np.sum(np.maximum(np.asarray(state, dtype=float), 0.0))) <= 0.0


In [ ]:
def simulate_phase(
    max_pcn: int,
    config: TransitionConfiguration,
    *,
    phase: str,
    duration_days: int,
    initial_state: np.ndarray,
    record_trajectory: bool,
    chromosomal_copy: int = 1,
    growth_profile: GrowthProfile | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray, dict]:
    if phase not in {"loss", "gain"}:
        raise ValueError("phase must be 'loss' or 'gain'.")

    growth = growth_rates(
        max_pcn,
        selected=(phase == "gain"),
        chromosomal_copy=chromosomal_copy,
        growth_profile=growth_profile,
    )
    transitions = transition_matrix(max_pcn, config)
    state = np.asarray(initial_state, dtype=float).copy()
    trajectory_frames, daily_rows = [], []
    start_summary = summarize_state(state, growth, chromosomal_copy=chromosomal_copy)
    final_prepassage_summary = start_summary

    for day_index in range(duration_days):
        if population_is_extinct(state):
            daily_rows.append({
                "phase_day": day_index + 1,
                "prepassage_density": 0.0,
                "prepassage_TCN": np.nan,
                "prepassage_TCN_signal": 0.0,
            })
            break

        times, states_over_time, prepassage_state = integrate_day(
            state, growth, transitions, record_trajectory=record_trajectory
        )
        final_prepassage_summary = summarize_state(
            prepassage_state, growth, chromosomal_copy=chromosomal_copy
        )
        daily_rows.append({
            "phase_day": day_index + 1,
            "prepassage_density": final_prepassage_summary["density"],
            "prepassage_TCN": final_prepassage_summary["TCN_per_density"],
            "prepassage_TCN_signal": final_prepassage_summary["TCN_signal"],
        })

        if record_trajectory:
            records = []
            for column, hour in enumerate(times):
                summary = summarize_state(
                    states_over_time[:, column], growth, chromosomal_copy=chromosomal_copy
                )
                records.append({
                    "phase_day": day_index + hour / PASSAGE_INTERVAL_HOURS,
                    "phase_time_hours": day_index * PASSAGE_INTERVAL_HOURS + hour,
                    "period": day_index + 1,
                    "passage_status": "within_period",
                    **summary,
                })
            frame = pd.DataFrame(records)
            if not frame.empty:
                frame.loc[frame.index[-1], "passage_status"] = "pre_passage"
                if day_index == 0:
                    frame.loc[frame.index[0], "passage_status"] = "phase_start"
                else:
                    frame = frame.iloc[1:].copy()
                trajectory_frames.append(frame)

        state = prepassage_state / PASSAGE_DILUTION_FACTOR
        if SUBPOPULATION_CUTOFF > 0:
            state[(state > 0) & (state < SUBPOPULATION_CUTOFF)] = 0.0

        if record_trajectory:
            post_summary = summarize_state(state, growth, chromosomal_copy=chromosomal_copy)
            trajectory_frames.append(pd.DataFrame([{
                "phase_day": day_index + 1.0,
                "phase_time_hours": (day_index + 1) * PASSAGE_INTERVAL_HOURS,
                "period": day_index + 1,
                "passage_status": "post_passage",
                **post_summary,
            }]))

    trajectory = pd.concat(trajectory_frames, ignore_index=True) if trajectory_frames else pd.DataFrame()
    daily = pd.DataFrame(daily_rows)
    for frame in (trajectory, daily):
        if not frame.empty:
            frame["phase"] = phase
            frame["condition"] = config.condition
            frame["maximum_PCN"] = int(max_pcn)
            frame["setting_label"] = config.setting_label
            frame["chromosomal_copy"] = chromosomal_copy

    summary = {
        "phase": phase,
        "condition": config.condition,
        "maximum_PCN": int(max_pcn),
        "setting_label": config.setting_label,
        "chromosomal_copy": chromosomal_copy,
        "initial_TCN": float(start_summary["TCN_per_density"]),
        "final_TCN": float(final_prepassage_summary["TCN_per_density"]),
        "final_prepassage_density": float(final_prepassage_summary["density"]),
        "minimum_daily_prepassage_density": float(daily["prepassage_density"].min()) if not daily.empty else 0.0,
        "postphase_boundary_density": float(np.sum(state)),
        "kappa_f": config.kappa_f,
        "kappa_b": config.kappa_b,
        "lower_forward_rate": config.lower_forward_rate,
        "population_extinct": population_is_extinct(state),
    }
    return trajectory, daily, state, summary


def simulate_transient_cycles(
    max_pcn: int,
    config: TransitionConfiguration,
    *,
    cycles: int = TRANSIENT_CYCLES,
    initial_state: np.ndarray | None = None,
    chromosomal_copy: int = 1,
    growth_profile: GrowthProfile | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    state = (
        periodic_initial_state(max_pcn, chromosomal_copy=chromosomal_copy)
        if initial_state is None else np.asarray(initial_state, dtype=float).copy()
    )
    trajectory_frames, daily_frames, cycle_rows = [], [], []

    for cycle in range(1, cycles + 1):
        loss_traj, loss_daily, state_after_loss, loss = simulate_phase(
            max_pcn, config, phase="loss", duration_days=RELAXED_DAYS_PER_CYCLE,
            initial_state=state, record_trajectory=True,
            chromosomal_copy=chromosomal_copy, growth_profile=growth_profile,
        )
        gain_traj, gain_daily, state_after_gain, gain = simulate_phase(
            max_pcn, config, phase="gain", duration_days=SELECTED_DAYS_PER_CYCLE,
            initial_state=state_after_loss, record_trajectory=True,
            chromosomal_copy=chromosomal_copy, growth_profile=growth_profile,
        )
        for phase_name, frame, day_offset in (
            ("loss", loss_traj, (cycle - 1) * DAYS_PER_CYCLE),
            ("gain", gain_traj, (cycle - 1) * DAYS_PER_CYCLE + RELAXED_DAYS_PER_CYCLE),
        ):
            frame = frame.copy()
            frame["cycle"] = cycle
            frame["phase"] = phase_name
            frame["Day"] = day_offset + frame["phase_day"]
            trajectory_frames.append(frame)
        for phase_name, frame, day_offset in (
            ("loss", loss_daily, (cycle - 1) * DAYS_PER_CYCLE),
            ("gain", gain_daily, (cycle - 1) * DAYS_PER_CYCLE + RELAXED_DAYS_PER_CYCLE),
        ):
            frame = frame.copy()
            frame["cycle"] = cycle
            frame["phase"] = phase_name
            frame["Day"] = day_offset + frame["phase_day"]
            daily_frames.append(frame)
        cycle_rows.append({
            "cycle": cycle,
            "condition": config.condition,
            "maximum_PCN": int(max_pcn),
            "setting_label": config.setting_label,
            "chromosomal_copy": chromosomal_copy,
            "TCN_low_end_relaxed": loss["final_TCN"],
            "TCN_high_end_selected": gain["final_TCN"],
            "density_end_relaxed": loss["final_prepassage_density"],
            "density_end_selected": gain["final_prepassage_density"],
            "minimum_daily_prepassage_density": min(
                loss["minimum_daily_prepassage_density"],
                gain["minimum_daily_prepassage_density"],
            ),
        })
        state = state_after_gain

    return (
        pd.concat(trajectory_frames, ignore_index=True),
        pd.concat(daily_frames, ignore_index=True),
        pd.DataFrame(cycle_rows),
    )


## Periodic convergence and response rates

Periodic convergence requires all three cycle-to-cycle criteria to hold for eight consecutive cycles, after at least 60 cycles. Three additional cycles are then simulated, and the final cycle is used for the reported response rates. Runs with minimum daily pre-passage total scaled density below 0.1 are recorded as missing for rate plots.


In [ ]:
def absolute_log_difference(current, previous) -> float:
    if previous is None or not np.isfinite(current) or not np.isfinite(previous):
        return np.inf
    if current <= 0 or previous <= 0:
        return np.inf
    return float(abs(np.log(current / previous)))


def advance_one_cycle(
    max_pcn: int,
    config: TransitionConfiguration,
    state: np.ndarray,
    *,
    chromosomal_copy: int = 1,
    growth_profile: GrowthProfile | None = None,
):
    _, _, state_after_loss, loss = simulate_phase(
        max_pcn, config, phase="loss", duration_days=RELAXED_DAYS_PER_CYCLE,
        initial_state=state, record_trajectory=False,
        chromosomal_copy=chromosomal_copy, growth_profile=growth_profile,
    )
    _, _, state_after_gain, gain = simulate_phase(
        max_pcn, config, phase="gain", duration_days=SELECTED_DAYS_PER_CYCLE,
        initial_state=state_after_loss, record_trajectory=False,
        chromosomal_copy=chromosomal_copy, growth_profile=growth_profile,
    )
    return state_after_gain, loss, gain


def converge_periodic_state(
    max_pcn: int,
    config: TransitionConfiguration,
    initial_state: np.ndarray,
    *,
    chromosomal_copy: int = 1,
    growth_profile: GrowthProfile | None = None,
) -> dict:
    state = np.asarray(initial_state, dtype=float).copy()
    previous_composition = normalized_composition(state)
    previous_boundary_density = float(state.sum())
    previous_low = previous_high = None
    stable = 0
    history = []
    low = high = minimum_density = np.nan

    for cycle in range(1, MAX_CONVERGENCE_CYCLES + 1):
        state, loss, gain = advance_one_cycle(
            max_pcn, config, state,
            chromosomal_copy=chromosomal_copy, growth_profile=growth_profile,
        )
        extinct = bool(loss["population_extinct"] or gain["population_extinct"])
        low, high = float(loss["final_TCN"]), float(gain["final_TCN"])
        boundary_density = float(state.sum())
        composition = normalized_composition(state)
        minimum_density = min(
            float(loss["minimum_daily_prepassage_density"]),
            float(gain["minimum_daily_prepassage_density"]),
        )

        composition_error = (
            np.inf if extinct or not np.isfinite(composition).all()
            else float(np.sum(np.abs(composition - previous_composition)))
        )
        boundary_error = absolute_log_difference(boundary_density, previous_boundary_density)
        low_error = absolute_log_difference(low, previous_low)
        high_error = absolute_log_difference(high, previous_high)
        endpoint_error = max(low_error, high_error)

        criteria_met = (
            not extinct
            and composition_error < COMPOSITION_CONVERGENCE_TOLERANCE
            and boundary_error < BOUNDARY_DENSITY_LOG_CONVERGENCE_TOLERANCE
            and endpoint_error < ENDPOINT_TCN_LOG_CONVERGENCE_TOLERANCE
        )
        stable = stable + 1 if criteria_met else 0
        history.append({
            "cycle": cycle,
            "condition": config.condition,
            "maximum_PCN": int(max_pcn),
            "chromosomal_copy": chromosomal_copy,
            "composition_L1_error": composition_error,
            "boundary_density_log_error": boundary_error,
            "endpoint_TCN_log_error": endpoint_error,
            "TCN_low": low,
            "TCN_high": high,
            "boundary_density": boundary_density,
            "minimum_daily_prepassage_density": minimum_density,
            "population_extinct": extinct,
            "consecutive_criteria_count": stable,
        })

        if extinct:
            return {
                "converged": False,
                "population_extinct": True,
                "state": state,
                "cycles_to_convergence": cycle,
                "TCN_low": low,
                "TCN_high": high,
                "minimum_daily_prepassage_density": minimum_density,
                "history": pd.DataFrame(history),
            }
        if cycle >= MIN_CONVERGENCE_CYCLES and stable >= CONSECUTIVE_CONVERGED_CYCLES:
            return {
                "converged": True,
                "population_extinct": False,
                "state": state,
                "cycles_to_convergence": cycle,
                "TCN_low": low,
                "TCN_high": high,
                "minimum_daily_prepassage_density": minimum_density,
                "history": pd.DataFrame(history),
            }

        previous_composition = composition
        previous_boundary_density = boundary_density
        previous_low, previous_high = low, high

    return {
        "converged": False,
        "population_extinct": False,
        "state": state,
        "cycles_to_convergence": MAX_CONVERGENCE_CYCLES,
        "TCN_low": low,
        "TCN_high": high,
        "minimum_daily_prepassage_density": minimum_density,
        "history": pd.DataFrame(history),
    }


def periodic_metrics(low_tcn, high_tcn, minimum_daily_prepassage_density, *, converged, population_extinct):
    survived = bool(
        np.isfinite(minimum_daily_prepassage_density)
        and minimum_daily_prepassage_density >= TOTAL_DENSITY_SURVIVAL_THRESHOLD
    )
    valid = bool(
        converged and not population_extinct and survived
        and np.isfinite(low_tcn) and np.isfinite(high_tcn)
        and low_tcn > 0 and high_tcn > 0
    )
    if not valid:
        return {
            "cycle_amplitude": np.nan,
            "loss_rate": np.nan,
            "gain_rate": np.nan,
            "survived_protocol": survived,
            "response_valid": False,
        }
    amplitude = float(np.log(high_tcn / low_tcn))
    return {
        "cycle_amplitude": amplitude,
        "loss_rate": amplitude / (RELAXED_DAYS_PER_CYCLE * PASSAGE_GENERATIONS),
        "gain_rate": amplitude / (SELECTED_DAYS_PER_CYCLE * PASSAGE_GENERATIONS),
        "survived_protocol": True,
        "response_valid": True,
    }


def evaluate_configuration(
    max_pcn: int,
    config: TransitionConfiguration,
    initial_state: np.ndarray | None = None,
    *,
    chromosomal_copy: int = 1,
    growth_profile: GrowthProfile | None = None,
) -> dict:
    if initial_state is None:
        initial_state = periodic_initial_state(max_pcn, chromosomal_copy=chromosomal_copy)
    result = converge_periodic_state(
        max_pcn, config, initial_state,
        chromosomal_copy=chromosomal_copy, growth_profile=growth_profile,
    )
    metrics = periodic_metrics(
        result["TCN_low"], result["TCN_high"], result["minimum_daily_prepassage_density"],
        converged=result["converged"], population_extinct=result["population_extinct"],
    )
    return {
        "condition": config.condition,
        "setting_label": config.setting_label,
        "maximum_PCN": int(max_pcn),
        "chromosomal_copy": chromosomal_copy,
        "kappa_f": config.kappa_f,
        "kappa_b": config.kappa_b,
        "converged": result["converged"],
        "population_extinct": result["population_extinct"],
        "cycles_to_convergence": result["cycles_to_convergence"],
        "TCN_low": result["TCN_low"] if metrics["response_valid"] else np.nan,
        "TCN_high": result["TCN_high"] if metrics["response_valid"] else np.nan,
        "minimum_daily_prepassage_density": result["minimum_daily_prepassage_density"],
        **metrics,
    }


def simulate_post_convergence_cycles(
    max_pcn: int,
    config: TransitionConfiguration,
    converged_state: np.ndarray,
    *,
    chromosomal_copy: int = 1,
    growth_profile: GrowthProfile | None = None,
) -> tuple[pd.DataFrame, dict]:
    state = np.asarray(converged_state, dtype=float).copy()
    frames, rows = [], []
    for cycle in range(1, ADDITIONAL_CYCLES_AFTER_CONVERGENCE + 1):
        loss_traj, _, state_after_loss, loss = simulate_phase(
            max_pcn, config, phase="loss", duration_days=RELAXED_DAYS_PER_CYCLE,
            initial_state=state, record_trajectory=True,
            chromosomal_copy=chromosomal_copy, growth_profile=growth_profile,
        )
        gain_traj, _, state_after_gain, gain = simulate_phase(
            max_pcn, config, phase="gain", duration_days=SELECTED_DAYS_PER_CYCLE,
            initial_state=state_after_loss, record_trajectory=True,
            chromosomal_copy=chromosomal_copy, growth_profile=growth_profile,
        )
        for phase_name, frame, offset in (
            ("loss", loss_traj, (cycle - 1) * DAYS_PER_CYCLE),
            ("gain", gain_traj, (cycle - 1) * DAYS_PER_CYCLE + RELAXED_DAYS_PER_CYCLE),
        ):
            frame = frame.copy()
            frame["cycle"] = cycle
            frame["phase"] = phase_name
            frame["Day"] = offset + frame["phase_day"]
            frames.append(frame)
        minimum_density = min(
            loss["minimum_daily_prepassage_density"], gain["minimum_daily_prepassage_density"]
        )
        metrics = periodic_metrics(
            loss["final_TCN"], gain["final_TCN"], minimum_density,
            converged=True,
            population_extinct=bool(loss["population_extinct"] or gain["population_extinct"]),
        )
        rows.append({
            "cycle_after_convergence": cycle,
            "condition": config.condition,
            "maximum_PCN": int(max_pcn),
            "chromosomal_copy": chromosomal_copy,
            "TCN_low": loss["final_TCN"],
            "TCN_high": gain["final_TCN"],
            "minimum_daily_prepassage_density": minimum_density,
            **metrics,
        })
        state = state_after_gain
    table = pd.DataFrame(rows)
    return pd.concat(frames, ignore_index=True), {
        "cycle_table": table,
        "reported": table.iloc[-1].to_dict(),
        "final_state": state,
    }


## Parameters used by the primary simulations

The values below are the shared values. Figure-specific sensitivity notebooks state the quantities they change.


In [ ]:
MODEL_PARAMETER_TABLE = pd.DataFrame([
    ("mu_max", MODEL.mu_max, "h^-1"),
    ("mu_infinity", MODEL.mu_infinity, "h^-1"),
    ("n_c", MODEL.burden_scale, ""),
    ("delta_A", MODEL.delta_antibiotic, "h^-1"),
    ("gamma", MODEL.gamma, ""),
    ("alpha", MODEL.alpha, ""),
    ("h", MODEL.hill, ""),
    ("A_relaxed", MODEL.antibiotic_relaxed, ""),
    ("A_selected", MODEL.antibiotic_selected, ""),
], columns=["parameter", "value", "units"])

TRANSITION_PARAMETER_TABLE = pd.DataFrame([
    ("transposition off", TRANSITIONS.kappa_f_off, TRANSITIONS.kappa_b_off),
    ("transposition on", TRANSITIONS.kappa_f_on, TRANSITIONS.kappa_b_on),
    ("transposition on + PCN dynamics", TRANSITIONS.kappa_f_pcn, TRANSITIONS.kappa_b_pcn),
], columns=["configuration", "kappa_f_h^-1", "kappa_b_h^-1"])
TRANSITION_PARAMETER_TABLE["lower_forward_rate_h^-1"] = TRANSITIONS.lower_forward_rate

PROTOCOL_PARAMETER_TABLE = pd.DataFrame([
    ("passage interval", PASSAGE_INTERVAL_HOURS, "h"),
    ("dilution factor", PASSAGE_DILUTION_FACTOR, "fold"),
    ("generation equivalents per passage", PASSAGE_GENERATIONS, "gen"),
    ("relaxed passages per cycle", RELAXED_DAYS_PER_CYCLE, "days"),
    ("selected passages per cycle", SELECTED_DAYS_PER_CYCLE, "days"),
    ("survival threshold", TOTAL_DENSITY_SURVIVAL_THRESHOLD, "scaled density"),
], columns=["parameter", "value", "units"])


def make_output_dirs(root_name: str):
    root = Path(root_name)
    fig_dir = root / "figures"
    source_dir = root / "source_data"
    fig_dir.mkdir(parents=True, exist_ok=True)
    source_dir.mkdir(parents=True, exist_ok=True)
    return root, fig_dir, source_dir


def save_figure(fig, stem: str, fig_dir: Path) -> None:
    for suffix in ("pdf", "svg", "png"):
        fig.savefig(fig_dir / f"{stem}.{suffix}", bbox_inches="tight", facecolor="white")


def export_table(table: pd.DataFrame, filename: str, source_dir: Path) -> None:
    table.to_csv(source_dir / filename, index=False)


def core_metadata(figure_id: str) -> pd.DataFrame:
    return pd.DataFrame([{
        "figure": figure_id,
        "core_version": CORE_VERSION,
        "parameter_manifest_version": PARAMETER_MANIFEST_VERSION,
        "run_mode": RUN_MODE,
        "python_version": platform.python_version(),
        "max_step_hours": MAX_STEP_HOURS,
        "output_step_hours": OUTPUT_STEP_HOURS,
        "atol": ATOL,
        "rtol": RTOL,
        "negative_state_tolerance": NEGATIVE_STATE_TOLERANCE,
        "composition_convergence_tolerance": COMPOSITION_CONVERGENCE_TOLERANCE,
        "boundary_density_log_convergence_tolerance": BOUNDARY_DENSITY_LOG_CONVERGENCE_TOLERANCE,
        "endpoint_TCN_log_convergence_tolerance": ENDPOINT_TCN_LOG_CONVERGENCE_TOLERANCE,
        "consecutive_converged_cycles": CONSECUTIVE_CONVERGED_CYCLES,
        "minimum_convergence_cycles": MIN_CONVERGENCE_CYCLES,
        "maximum_convergence_cycles": MAX_CONVERGENCE_CYCLES,
        "additional_cycles_after_convergence": ADDITIONAL_CYCLES_AFTER_CONVERGENCE,
        "survival_threshold": TOTAL_DENSITY_SURVIVAL_THRESHOLD,
    }])
